In [99]:
import pandas as pd

In [100]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

client_id = "ab30884cbf2b41fda2e26aab4e3a7751"
client_secret = "fae12b6886be489eb8a8c8135978e8bb"

In [101]:

auth_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)

sp = spotipy.Spotify(auth_manager=auth_manager)

print("Connected to Spotify API")

Connected to Spotify API


In [102]:
playlist_id = "7zXhVAENGtOlG6Mnwk7bwv"


In [103]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id="ab30884cbf2b41fda2e26aab4e3a7751",
    client_secret="fae12b6886be489eb8a8c8135978e8bb",
    redirect_uri="http://127.0.0.1:8888/callback"
))

In [104]:
results = sp.playlist_items("7zXhVAENGtOlG6Mnwk7bwv")

print(results)

{'href': 'https://api.spotify.com/v1/playlists/7zXhVAENGtOlG6Mnwk7bwv/items?offset=0&limit=50&additional_types=track', 'items': [{'added_at': '2025-04-04T19:09:42Z', 'added_by': {'external_urls': {'spotify': 'https://open.spotify.com/user/31jlytj4bkpsqaksxdmzbvhoyccm'}, 'href': 'https://api.spotify.com/v1/users/31jlytj4bkpsqaksxdmzbvhoyccm', 'id': '31jlytj4bkpsqaksxdmzbvhoyccm', 'type': 'user', 'uri': 'spotify:user:31jlytj4bkpsqaksxdmzbvhoyccm'}, 'is_local': False, 'primary_color': None, 'item': {'is_playable': True, 'explicit': False, 'type': 'track', 'episode': False, 'track': True, 'album': {'is_playable': True, 'type': 'album', 'album_type': 'album', 'href': 'https://api.spotify.com/v1/albums/0zV96rKdfWliVHNBpAsd2b', 'id': '0zV96rKdfWliVHNBpAsd2b', 'images': [{'height': 640, 'url': 'https://i.scdn.co/image/ab67616d0000b273ef759d4ae310020a06939e99', 'width': 640}, {'height': 300, 'url': 'https://i.scdn.co/image/ab67616d00001e02ef759d4ae310020a06939e99', 'width': 300}, {'height': 64,

In [105]:
results = sp.playlist_items("7zXhVAENGtOlG6Mnwk7bwv")


In [106]:
songs = []

for item in results["items"]:
    
    track = item.get("item")
    
    if track:
        
        artists = ", ".join([a["name"] for a in track["artists"]])
        
        songs.append({
            "track_name": track["name"],
            "album_name": track["album"]["name"],
            "artists": artists,
            "track_id": track["id"]
        })

playlist_df = pd.DataFrame(songs)

In [107]:
playlist_df = pd.DataFrame(songs)

print(playlist_df)

                                   track_name  \
0                                       Lover   
1                                      Softly   
2                                     Excuses   
3                                     Pasoori   
4                                    With You   
5                                      Dil Nu   
6                           White Brown Black   
7                               Bijlee Bijlee   
8                             Suniyan Suniyan   
9                                    G.O.A.T.   
10                        Naina (From "Crew")   
11                                  Blue Eyes   
12                                  Love Dose   
13                              Desi Kalakaar   
14                                 Brown Rang   
15                                Millionaire   
16  Dil Chori (From "Sonu Ke Titu Ki Sweety")   
17                                  Old Money   
18                                   One Love   
19                  

In [108]:


playlist_df.columns

Index(['track_name', 'album_name', 'artists', 'track_id'], dtype='object')

In [109]:
df = pd.read_csv("dataset.csv")

In [110]:
df["track_name"] = df["track_name"].str.lower()
df["album_name"] = df["album_name"].str.lower()
df["artists"] = df["artists"].str.lower()

playlist_df["track_name"] = playlist_df["track_name"].str.lower()
playlist_df["album_name"] = playlist_df["album_name"].str.lower()
playlist_df["artists"] = playlist_df["artists"].str.lower()

In [111]:
df = df.drop_duplicates(subset=["track_id"])

In [112]:
df["track_name"] = df["track_name"].str.lower()
playlist_df["track_name"] = playlist_df["track_name"].str.lower()

filtered_df = df[
    df.set_index(["track_name","album_name","artists"]).index.isin(
        playlist_df.set_index(["track_name","album_name","artists"]).index
    )
]

In [113]:
print(len(filtered_df))


12


In [114]:
print(len(songs))

50


In [117]:
filtered_df = filtered_df.drop_duplicates(subset=["track_id"])

In [118]:
filtered_df["track_name"].value_counts()

track_name
desi kalakaar                              1
love dose                                  1
blue eyes                                  1
dheere dheere                              1
brown rang                                 1
chaar botal vodka (from "ragini mms 2")    1
one bottle down                            1
lover                                      1
born to shine                              1
g.o.a.t.                                   1
na ja                                      1
bijlee bijlee                              1
Name: count, dtype: int64

In [119]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

In [120]:
playlist_features = filtered_df[features]

In [121]:
playlist_vector = playlist_features.mean().values.reshape(1,-1)

In [122]:
dataset_features = df[features]

In [123]:


features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

playlist_features = filtered_df[features]


dataset_features = df[features]



In [124]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(playlist_features, dataset_features)

In [125]:
import numpy as np

scores = similarity_matrix.mean(axis=0)

In [126]:
dataset_without_playlist = df[~df["track_id"].isin(filtered_df["track_id"])]

In [127]:
top_indices = np.argsort(scores)[::-1][:10]

recommended_songs = df.iloc[top_indices]

print(recommended_songs[["track_name","artists","album_name"]])

                                              track_name              artists  \
65650  all night (bts world original soundtrack) [pt. 3]       bts;juice wrld   
97555    se é pra falar de amor (tema novela a favorita)   mateus e cristiano   
35916                                  saudade né filha?           max e luan   
15339                                  forever right now       conor matthews   
1002                                             fellini               criolo   
30694                                          follow me   sam feldt;rita ora   
53853                        one more dance (with alida)          r3hab;alida   
83787                              perfect (feat. haris)  lucas & steve;haris   
83841                              perfect (feat. haris)  lucas & steve;haris   
99054                                           isabelle            zach hood   

                                              album_name  
65650  all night (bts world original soundtrack) 

---
## 🎵 Single Song Recommendation Mode
Instead of using a full playlist, you can recommend songs based on **one song**.
Just enter the song name (and optionally artist name) below.

In [ ]:
# ──────────────────────────────────────────────────────────
# SINGLE SONG RECOMMENDATION
# Enter the song name and (optionally) the artist below
# ──────────────────────────────────────────────────────────

SONG_NAME  = "Brown Munde"       # change this to any song
ARTIST_NAME = "AP Dhillon"       # leave empty string "" if unsure

# Search for the track on Spotify
query = f"track:{SONG_NAME}"
if ARTIST_NAME:
    query += f" artist:{ARTIST_NAME}"

search_results = sp.search(q=query, type="track", limit=5)
tracks = search_results["tracks"]["items"]

if not tracks:
    print("❌ Song not found. Try a different name.")
else:
    for i, t in enumerate(tracks):
        artists = ", ".join([a["name"] for a in t["artists"]])
        print(f"[{i}] {t['name']} — {artists}  (id: {t['id']})")

In [ ]:
# Pick the correct result index from the list above (default 0 = first result)
PICK_INDEX = 0

chosen_track = tracks[PICK_INDEX]
chosen_track_id = chosen_track["id"]
chosen_track_name = chosen_track["name"]
print(f"✅ Selected: {chosen_track_name} (id: {chosen_track_id})")

In [ ]:
# ── Match the chosen song in our dataset ──
features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

matched = df[df["track_id"] == chosen_track_id]

if matched.empty:
    print("⚠️  Song not found in the local dataset by track_id.")
    print("   Trying by name...")
    matched = df[df["track_name"].str.lower() == chosen_track_name.lower()]

if matched.empty:
    print("❌ Song not in dataset. Try a different song.")
else:
    print(f"✅ Found in dataset: {matched.iloc[0]['track_name']} — {matched.iloc[0]['artists']}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Use the first match
song_vector = matched.iloc[[0]][features].values  # shape (1, 9)

# Exclude the chosen song itself
other_songs = df[df["track_id"] != matched.iloc[0]["track_id"]].copy()

# Compute similarity
other_features = other_songs[features].values
sim_scores = cosine_similarity(song_vector, other_features)[0]   # shape (n,)

# Top 10
top_idx = np.argsort(sim_scores)[::-1][:10]
single_song_recs = other_songs.iloc[top_idx][["track_name", "artists", "album_name"]].reset_index(drop=True)

print(f"\n🎧 Top 10 songs similar to '{chosen_track_name}':\n")
print(single_song_recs.to_string(index=True))